In [ ]:

# Step 1: Environment check
import sys
import torch
import sklearn
import pandas as pd
import numpy as np

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("scikit-learn version:", sklearn.__version__)
print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)
print()
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected. Check Runtime > Change runtime type.")

Python version: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch version: 2.14.1+cu130
scikit-learn version: 1.6.1
pandas version: 2.2.3
numpy version: 2.4.6

GPU available: True
GPU name: Tesla T4


In [ ]:
# Step 2: Download PhiUSIIL from UCI and verify class counts
!pip install -q ucimlrepo

from ucimlrepo import fetch_ucirepo
import pandas as pd

# PhiUSIIL Phishing URL Dataset is UCI dataset id 967
phiusiil = fetch_ucirepo(id=967)

X = phiusiil.data.features
y = phiusiil.data.targets

df = pd.concat([X, y], axis=1)

print("=" * 60)
print("BASIC SHAPE")
print("=" * 60)
print("Total rows:", len(df))
print("Total columns:", df.shape[1])
print()

print("=" * 60)
print("ALL COLUMN NAMES")
print("=" * 60)
for i, col in enumerate(df.columns):
    print(f"{i:3d}  {col}")
print()

print("=" * 60)
print("LABEL COLUMN RAW COUNTS")
print("=" * 60)
label_col = y.columns[0]
print("Label column name:", label_col)
print(df[label_col].value_counts())
print()

print("=" * 60)
print("DUPLICATE CHECK")
print("=" * 60)
print("Duplicate rows:", df.duplicated().sum())

BASIC SHAPE
Total rows: 235795
Total columns: 55

ALL COLUMN NAMES
  0  URL
  1  URLLength
  2  Domain
  3  DomainLength
  4  IsDomainIP
  5  TLD
  6  URLSimilarityIndex
  7  CharContinuationRate
  8  TLDLegitimateProb
  9  URLCharProb
 10  TLDLength
 11  NoOfSubDomain
 12  HasObfuscation
 13  NoOfObfuscatedChar
 14  ObfuscationRatio
 15  NoOfLettersInURL
 16  LetterRatioInURL
 17  NoOfDegitsInURL
 18  DegitRatioInURL
 19  NoOfEqualsInURL
 20  NoOfQMarkInURL
 21  NoOfAmpersandInURL
 22  NoOfOtherSpecialCharsInURL
 23  SpacialCharRatioInURL
 24  IsHTTPS
 25  LineOfCode
 26  LargestLineLength
 27  HasTitle
 28  Title
 29  DomainTitleMatchScore
 30  URLTitleMatchScore
 31  HasFavicon
 32  Robots
 33  IsResponsive
 34  NoOfURLRedirect
 35  NoOfSelfRedirect
 36  HasDescription
 37  NoOfPopup
 38  NoOfiFrame
 39  HasExternalFormSubmit
 40  HasSocialNet
 41  HasSubmitButton
 42  HasHiddenFields
 43  HasPasswordField
 44  Bank
 45  Pay
 46  Crypto
 47  HasCopyrightInfo
 48  NoOfImage
 49  NoOf

In [ ]:
# Step 2b: Verify which label value means phishing
import pandas as pd
pd.set_option("display.max_colwidth", 100)

print("=" * 60)
print("SAMPLE URLS, label = 1  (first 15)")
print("=" * 60)
print(df[df["label"] == 1]["URL"].head(15).to_string())
print()

print("=" * 60)
print("SAMPLE URLS, label = 0  (first 15)")
print("=" * 60)
print(df[df["label"] == 0]["URL"].head(15).to_string())
print()

print("=" * 60)
print("WELL-KNOWN DOMAIN CHECK")
print("=" * 60)
known = ["google.com", "wikipedia.org", "youtube.com", "microsoft.com", "amazon.com"]
for dom in known:
    hits = df[df["Domain"].str.contains(dom, case=False, na=False)]
    if len(hits) > 0:
        print(f"{dom}: {len(hits)} rows, label distribution: {hits['label'].value_counts().to_dict()}")
    else:
        print(f"{dom}: not found")
print()

print("=" * 60)
print("TRUST-LIKE FEATURE MEANS BY LABEL")
print("=" * 60)
for col in ["TLDLegitimateProb", "IsHTTPS", "URLSimilarityIndex", "HasObfuscation"]:
    means = df.groupby("label")[col].mean()
    print(f"{col}:  label 0 = {means[0]:.4f}   label 1 = {means[1]:.4f}")

SAMPLE URLS, label = 1  (first 15)
0       https://www.southbankmosaics.com
1               https://www.uni-mainz.de
2         https://www.voicefmradio.co.uk
3            https://www.sfnmjournal.com
4     https://www.rewildingargentina.org
5        https://www.globalreporting.org
6             https://www.saffronart.com
7             https://www.nerdscandy.com
8         https://www.hyderabadonline.in
9                    https://www.aap.org
10    https://www.religionenlibertad.com
12          https://www.socialpolicy.org
13                 https://www.aoh61.com
14           https://www.bulgariaski.com
15             https://www.brightika.com

SAMPLE URLS, label = 0  (first 15)
11                                                                                http://www.teramill.com
20                                                                            http://www.f0519141.xsph.ru
21                                                                               http://www.shprakserf

In [ ]:

# Step 2c: Save dataset to Google Drive
from google.colab import drive
drive.mount("/content/drive")

import os
save_dir = "/content/drive/MyDrive/thesis_experiments"
os.makedirs(save_dir, exist_ok=True)

df.to_parquet(f"{save_dir}/phiusiil_full.parquet")
print("Saved:", os.listdir(save_dir))
print("File size (MB):", round(os.path.getsize(f"{save_dir}/phiusiil_full.parquet") / 1e6, 1))

Mounted at /content/drive
Saved: ['split_val.parquet', 'split_test.parquet', 'feature_ranking.csv', 'scaler_min_F.npy', 'scaler_rng_F.npy', 'baselineF_seed42.pt', 'baselineF_seed43.pt', 'baselineF_seed44.pt', 'baselineF_test_results.csv', 'scaler_min_C.npy', 'scaler_rng_C.npy', 'baselineC_seed42.pt', 'baselineC_seed43.pt', 'baselineC_seed44.pt', 'baselineC_test_results.csv', 'scaler_min_U.npy', 'scaler_rng_U.npy', 'features_U.txt', 'baselineU_seed42.pt', 'baselineU_seed43.pt', 'baselineU_seed44.pt', 'baselineU_test_results.csv', 'pruneU_s50_seed42.pt', 'pruneU_s70_seed42.pt', 'pruneU_s90_seed42.pt', 'pruneU_s95_seed42.pt', 'pruneU_s50_seed43.pt', 'pruneU_s70_seed43.pt', 'pruneU_s90_seed43.pt', 'pruneU_s95_seed43.pt', 'pruneU_s50_seed44.pt', 'pruneU_s70_seed44.pt', 'pruneU_s90_seed44.pt', 'pruneU_s95_seed44.pt', 'pruning_grid_results.csv', 'pruneU_iter_s70_seed42.pt', 'pruneU_iter_s90_seed42.pt', 'pruneU_iter_s95_seed42.pt', 'pruneU_iter_s70_seed43.pt', 'pruneU_iter_s90_seed43.pt', 'pru

In [ ]:
# Step 3: Chronological split (row-order proxy) and integrity checks
import pandas as pd

df = pd.read_parquet("/content/drive/MyDrive/thesis_experiments/phiusiil_full.parquet")

string_cols = ["URL", "Domain", "TLD", "Title"]
feature_cols = [c for c in df.columns if c not in string_cols + ["label"]]
print("Numeric feature count:", len(feature_cols))
print()

n = len(df)
i_train = int(n * 0.70)
i_val = int(n * 0.85)

train = df.iloc[:i_train].copy()
val   = df.iloc[i_train:i_val].copy()
test  = df.iloc[i_val:].copy()

print("Split sizes:", len(train), len(val), len(test))
print()

print("Class balance per split (fraction phishing, label 0):")
for name, part in [("train", train), ("val", val), ("test", test)]:
    frac = (part["label"] == 0).mean()
    print(f"  {name}: {frac:.4f}   (n = {len(part)})")
print()

# Domain overlap: how many test domains were already seen in training?
train_domains = set(train["Domain"].str.lower())
test_domains = set(test["Domain"].str.lower())
overlap = test_domains & train_domains
print("Unique domains in train:", len(train_domains))
print("Unique domains in test:", len(test_domains))
print(f"Test domains also in train: {len(overlap)} ({len(overlap)/len(test_domains)*100:.1f}%)")
print()

# Same check restricted to phishing test rows (the zero-day claim)
test_phish_domains = set(test[test["label"] == 0]["Domain"].str.lower())
phish_overlap = test_phish_domains & train_domains
print("Unique phishing domains in test:", len(test_phish_domains))
print(f"Of those, seen in train: {len(phish_overlap)} ({len(phish_overlap)/len(test_phish_domains)*100:.1f}%)")

# Save splits
save_dir = "/content/drive/MyDrive/thesis_experiments"
train.to_parquet(f"{save_dir}/split_train.parquet")
val.to_parquet(f"{save_dir}/split_val.parquet")
test.to_parquet(f"{save_dir}/split_test.parquet")
print()
print("Splits saved to Drive.")

Numeric feature count: 50

Split sizes: 165056 35369 35370

Class balance per split (fraction phishing, label 0):
  train: 0.4268   (n = 165056)
  val: 0.4315   (n = 35369)
  test: 0.4306   (n = 35370)

Unique domains in train: 155581
Unique domains in test: 33919
Test domains also in train: 1543 (4.5%)

Unique phishing domains in test: 13780
Of those, seen in train: 1536 (11.1%)

Splits saved to Drive.


In [ ]:
# Step 4: Hybrid feature ranking (train data only) + leakage check
import pandas as pd
import numpy as np
from sklearn.feature_selection import mutual_info_classif

save_dir = "/content/drive/MyDrive/thesis_experiments"
train = pd.read_parquet(f"{save_dir}/split_train.parquet")

string_cols = ["URL", "Domain", "TLD", "Title"]
feature_cols = [c for c in train.columns if c not in string_cols + ["label"]]

X = train[feature_cols].astype(float)
y = train["label"].values

# 1. Statistical correlation with the label (paper's "statistical correlation")
corr_scores = X.apply(lambda col: abs(np.corrcoef(col, y)[0, 1]))

# 2. Information gain (mutual information)
print("Computing mutual information (takes a minute or two)...")
mi = mutual_info_classif(X, y, discrete_features=False, random_state=42)
mi_scores = pd.Series(mi, index=feature_cols)

# 3. Hybrid score: alpha-weighted combination as in the paper (alpha = 0.5)
corr_n = corr_scores / corr_scores.max()
mi_n = mi_scores / mi_scores.max()
hybrid = 0.5 * corr_n + 0.5 * mi_n

ranking = pd.DataFrame({
    "abs_corr": corr_scores,
    "mutual_info": mi_scores,
    "hybrid_score": hybrid
}).sort_values("hybrid_score", ascending=False)

print()
print("TOP 15 FEATURES BY HYBRID SCORE")
print(ranking.head(15).round(4).to_string())
print()
print("BOTTOM 10 FEATURES")
print(ranking.tail(10).round(4).to_string())
print()

# 4. Redundancy: feature pairs with |corr| > 0.9 (paper's correlation filter)
cm = X.corr().abs()
pairs = []
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        if cm.iloc[i, j] > 0.9:
            pairs.append((feature_cols[i], feature_cols[j], round(cm.iloc[i, j], 3)))
print(f"HIGHLY CORRELATED PAIRS (|r| > 0.9): {len(pairs)}")
for a, b, r in pairs:
    print(f"  {a} <-> {b}: {r}")
print()

# 5. Leakage check on the top feature(s): how well does each single
#    feature separate the classes on its own?
print("SINGLE-FEATURE SEPARATION CHECK (top 5 by hybrid score)")
for feat in ranking.head(5).index:
    phish_mean = X.loc[y == 0, feat].mean()
    legit_mean = X.loc[y == 1, feat].mean()
    # simple threshold accuracy using the midpoint
    thr = (phish_mean + legit_mean) / 2
    if legit_mean > phish_mean:
        pred = (X[feat] > thr).astype(int)
    else:
        pred = (X[feat] <= thr).astype(int)
    acc = (pred == y).mean()
    print(f"  {feat}: phishing mean {phish_mean:.3f}, legit mean {legit_mean:.3f}, "
          f"single-feature threshold accuracy {acc:.4f}")

ranking.to_csv(f"{save_dir}/feature_ranking.csv")
print()
print("Ranking saved to Drive.")

Computing mutual information (takes a minute or two)...

TOP 15 FEATURES BY HYBRID SCORE
                       abs_corr  mutual_info  hybrid_score
URLSimilarityIndex       0.8591       0.6743        1.0000
HasSocialNet             0.7839       0.4075        0.7584
HasCopyrightInfo         0.7442       0.3430        0.6874
HasDescription           0.6898       0.2970        0.6217
LineOfCode               0.2641       0.6041        0.6017
NoOfSelfRef              0.3517       0.5263        0.5950
NoOfExternalRef          0.2859       0.5632        0.5840
NoOfJS                   0.3384       0.5023        0.5695
NoOfImage                0.2721       0.5450        0.5625
IsHTTPS                  0.6175       0.2553        0.5487
HasSubmitButton          0.5810       0.2057        0.4907
DomainTitleMatchScore    0.5809       0.2054        0.4904
LetterRatioInURL         0.3613       0.3763        0.4893
IsResponsive             0.5544       0.1836        0.4588
SpacialCharRatioInURL    0

In [ ]:

# Step 5a: Preprocessing + ADUIN model definition + sanity check
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

save_dir = "/content/drive/MyDrive/thesis_experiments"
train = pd.read_parquet(f"{save_dir}/split_train.parquet")
val   = pd.read_parquet(f"{save_dir}/split_val.parquet")
test  = pd.read_parquet(f"{save_dir}/split_test.parquet")

string_cols = ["URL", "Domain", "TLD", "Title"]
FEATURES_F = [c for c in train.columns if c not in string_cols + ["label"]]  # 50, faithful
FEATURES_C = [c for c in FEATURES_F if c != "URLSimilarityIndex"]            # 49, controlled

def make_arrays(feature_list):
    Xtr = train[feature_list].astype(np.float32).values
    Xva = val[feature_list].astype(np.float32).values
    Xte = test[feature_list].astype(np.float32).values
    # Min-max scaling fitted on TRAIN ONLY
    mn, mx = Xtr.min(axis=0), Xtr.max(axis=0)
    rng = np.where((mx - mn) == 0, 1.0, mx - mn)
    Xtr = (Xtr - mn) / rng
    Xva = np.clip((Xva - mn) / rng, 0.0, 1.0)
    Xte = np.clip((Xte - mn) / rng, 0.0, 1.0)
    ytr = train["label"].values.astype(np.float32)
    yva = val["label"].values.astype(np.float32)
    yte = test["label"].values.astype(np.float32)
    return (Xtr, ytr), (Xva, yva), (Xte, yte)

class ADUIN(nn.Module):
    def __init__(self, n_features, d_model=128, n_heads=8, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout),
        )
        self.out = nn.Linear(32, 1)

    def forward(self, x):
        h = self.input_proj(x)            # (batch, d_model)
        h = h.unsqueeze(1)                # (batch, 1, d_model) as a length-1 sequence
        a, _ = self.attn(h, h, h)         # multi-head self-attention
        h = self.norm(h + a).squeeze(1)   # residual + norm
        h = self.mlp(h)
        return self.out(h).squeeze(-1)    # raw logit; sigmoid applied in the loss

# Sanity check on the faithful feature set
(Xtr, ytr), (Xva, yva), (Xte, yte) = make_arrays(FEATURES_F)
print("Shapes:", Xtr.shape, Xva.shape, Xte.shape)
print("Scaled train range: [", Xtr.min(), ",", Xtr.max(), "]")

device = "cuda" if torch.cuda.is_available() else "cpu"
model = ADUIN(n_features=50).to(device)

n_params = sum(p.numel() for p in model.parameters())
size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6
print(f"Parameter count: {n_params:,}")
print(f"Model size (FP32, MB): {size_mb:.2f}")

# Forward pass on a small batch to confirm wiring
xb = torch.tensor(Xtr[:8]).to(device)
with torch.no_grad():
    logits = model(xb)
print("Forward pass OK. Sample logits:", logits.cpu().numpy().round(3))

Shapes: (165056, 50) (35369, 50) (35370, 50)
Scaled train range: [ 0.0 , 1.0 ]
Parameter count: 99,713
Model size (FP32, MB): 0.40
Forward pass OK. Sample logits: [-0.037 -0.031 -0.043 -0.016 -0.004  0.011 -0.114  0.004]


In [ ]:
# Step 5b: Train Baseline F (50 features), 3 seeds
import pandas as pd, numpy as np, torch, torch.nn as nn, time, os

save_dir = "/content/drive/MyDrive/thesis_experiments"
train = pd.read_parquet(f"{save_dir}/split_train.parquet")
val   = pd.read_parquet(f"{save_dir}/split_val.parquet")

string_cols = ["URL", "Domain", "TLD", "Title"]
FEATURES_F = [c for c in train.columns if c not in string_cols + ["label"]]

Xtr = train[FEATURES_F].astype(np.float32).values
Xva = val[FEATURES_F].astype(np.float32).values
mn, mx = Xtr.min(axis=0), Xtr.max(axis=0)
rng = np.where((mx - mn) == 0, 1.0, mx - mn)
Xtr = (Xtr - mn) / rng
Xva = np.clip((Xva - mn) / rng, 0.0, 1.0)
ytr = train["label"].values.astype(np.float32)
yva = val["label"].values.astype(np.float32)
np.save(f"{save_dir}/scaler_min_F.npy", mn); np.save(f"{save_dir}/scaler_rng_F.npy", rng)

class ADUIN(nn.Module):
    def __init__(self, n_features, d_model=128, n_heads=8, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout),
        )
        self.out = nn.Linear(32, 1)
    def forward(self, x):
        h = self.input_proj(x).unsqueeze(1)
        a, _ = self.attn(h, h, h)
        h = self.norm(h + a).squeeze(1)
        return self.out(self.mlp(h)).squeeze(-1)

device = "cuda"
# Class weights from VERIFIED counts: minority (phishing, label 0) up-weighted.
frac_phish = (ytr == 0).mean()
w_pos = frac_phish          # weight applied to label-1 (legit) examples
w_neg = 1.0 - frac_phish    # weight applied to label-0 (phishing) examples
print(f"Verified train fractions: phishing {frac_phish:.4f}, legit {1-frac_phish:.4f}")
print(f"Loss weights: phishing errors x{w_neg:.3f}... wait, check below.")
# Clearer: per-sample weight = w where w = (1 - class_frequency of that sample's class)
sample_w_tr = np.where(ytr == 0, 1.0 - frac_phish, frac_phish).astype(np.float32)
print(f"Per-sample weights: phishing {1-frac_phish:.4f}, legit {frac_phish:.4f}")

def run_seed(seed, epochs=50, batch=128, patience=5):
    torch.manual_seed(seed); np.random.seed(seed)
    model = ADUIN(len(FEATURES_F)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    Xt = torch.tensor(Xtr); yt = torch.tensor(ytr); wt = torch.tensor(sample_w_tr)
    Xv = torch.tensor(Xva).to(device); yv = torch.tensor(yva).to(device)
    n = len(Xt); best_val = float("inf"); best_ep = 0; wait = 0
    ckpt = f"{save_dir}/baselineF_seed{seed}.pt"
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(n)
        tot = 0.0
        for i in range(0, n, batch):
            idx = perm[i:i+batch]
            xb, yb, wb = Xt[idx].to(device), yt[idx].to(device), wt[idx].to(device)
            opt.zero_grad()
            loss = nn.functional.binary_cross_entropy_with_logits(model(xb), yb, weight=wb)
            loss.backward(); opt.step()
            tot += loss.item() * len(idx)
        model.eval()
        with torch.no_grad():
            vl = nn.functional.binary_cross_entropy_with_logits(model(Xv), yv).item()
        if vl < best_val - 1e-5:
            best_val, best_ep, wait = vl, ep, 0
            torch.save(model.state_dict(), ckpt)
        else:
            wait += 1
            if wait >= patience:
                print(f"  seed {seed}: early stop at epoch {ep} (best epoch {best_ep})")
                break
        if ep % 10 == 0:
            print(f"  seed {seed} epoch {ep}: train loss {tot/n:.4f}, val loss {vl:.4f}")
    print(f"  seed {seed} done in {(time.time()-t0)/60:.1f} min, best val loss {best_val:.4f} at epoch {best_ep}")
    return best_ep, best_val

results = {}
for seed in [42, 43, 44]:
    print(f"Training seed {seed}...")
    results[seed] = run_seed(seed)

print()
print("All seeds finished. Best epochs and val losses:", results)

Verified train fractions: phishing 0.4268, legit 0.5732
Loss weights: phishing errors x0.573... wait, check below.
Per-sample weights: phishing 0.5732, legit 0.4268
Training seed 42...
  seed 42 epoch 10: train loss 0.0006, val loss 0.0003
  seed 42: early stop at epoch 15 (best epoch 10)
  seed 42 done in 1.6 min, best val loss 0.0003 at epoch 10
Training seed 43...
  seed 43: early stop at epoch 6 (best epoch 1)
  seed 43 done in 0.6 min, best val loss 0.0005 at epoch 1
Training seed 44...
  seed 44 epoch 10: train loss 0.0006, val loss 0.0004
  seed 44: early stop at epoch 13 (best epoch 8)
  seed 44 done in 1.4 min, best val loss 0.0002 at epoch 8

All seeds finished. Best epochs and val losses: {42: (10, 0.0002733799337875098), 43: (1, 0.0005033586639910936), 44: (8, 0.00024250365095213056)}


In [ ]:
# Step 5c: Evaluate Baseline F on the test set, all seeds
import pandas as pd, numpy as np, torch, torch.nn as nn
from sklearn.metrics import roc_auc_score

save_dir = "/content/drive/MyDrive/thesis_experiments"
test = pd.read_parquet(f"{save_dir}/split_test.parquet")

string_cols = ["URL", "Domain", "TLD", "Title"]
FEATURES_F = [c for c in test.columns if c not in string_cols + ["label"]]

mn = np.load(f"{save_dir}/scaler_min_F.npy"); rng = np.load(f"{save_dir}/scaler_rng_F.npy")
Xte = np.clip((test[FEATURES_F].astype(np.float32).values - mn) / rng, 0, 1).astype(np.float32)
yte = test["label"].values.astype(np.float32)

class ADUIN(nn.Module):
    def __init__(self, n_features, d_model=128, n_heads=8, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout),
        )
        self.out = nn.Linear(32, 1)
    def forward(self, x):
        h = self.input_proj(x).unsqueeze(1)
        a, _ = self.attn(h, h, h)
        h = self.norm(h + a).squeeze(1)
        return self.out(self.mlp(h)).squeeze(-1)

device = "cuda"
Xt = torch.tensor(Xte).to(device)

def evaluate(state_path):
    model = ADUIN(len(FEATURES_F)).to(device)
    model.load_state_dict(torch.load(state_path, map_location=device))
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(Xt)).cpu().numpy()  # prob of label 1 (legit)
    # Phishing is the positive class for detection metrics: p_phish = 1 - p_legit
    p_phish = 1 - probs
    pred_phish = (p_phish >= 0.5)
    actual_phish = (yte == 0)
    TP = np.sum(pred_phish & actual_phish)
    TN = np.sum(~pred_phish & ~actual_phish)
    FP = np.sum(pred_phish & ~actual_phish)
    FN = np.sum(~pred_phish & actual_phish)
    acc  = (TP + TN) / len(yte)
    prec = TP / (TP + FP) if TP + FP else 0.0
    rec  = TP / (TP + FN) if TP + FN else 0.0   # = zero-day detection rate here
    f1   = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    fpr  = FP / (FP + TN) if FP + TN else 0.0
    auc  = roc_auc_score(actual_phish.astype(int), p_phish)
    return dict(accuracy=acc, precision=prec, recall_zeroday=rec, f1=f1, fpr=fpr, auc=auc)

rows = []
for seed in [42, 43, 44]:
    m = evaluate(f"{save_dir}/baselineF_seed{seed}.pt")
    m["seed"] = seed
    rows.append(m)
res = pd.DataFrame(rows).set_index("seed")

print("PER-SEED RESULTS (test set, phishing = positive class)")
print((res * 100).round(2).to_string())
print()
summary = res.agg(["mean", "std"]) * 100
print("SUMMARY (percent)")
print(summary.round(2).to_string())
print()
print("PUBLISHED ADUIN FIGURES: accuracy 95, precision 93, zero-day 92, FPR 3.5")

res.to_csv(f"{save_dir}/baselineF_test_results.csv")
print()
print("Saved to baselineF_test_results.csv")

PER-SEED RESULTS (test set, phishing = positive class)
      accuracy  precision  recall_zeroday     f1  fpr    auc
seed                                                        
42       99.99     100.00           99.97  99.98  0.0  100.0
43       99.98      99.99           99.96  99.98  0.0  100.0
44       99.99     100.00           99.97  99.98  0.0  100.0

SUMMARY (percent)
      accuracy  precision  recall_zeroday     f1  fpr    auc
mean     99.98      100.0           99.96  99.98  0.0  100.0
std       0.00        0.0            0.00   0.00  0.0    0.0

PUBLISHED ADUIN FIGURES: accuracy 95, precision 93, zero-day 92, FPR 3.5

Saved to baselineF_test_results.csv


In [ ]:
# Step 6: Baseline C (49 features, URLSimilarityIndex removed): train + evaluate
import pandas as pd, numpy as np, torch, torch.nn as nn, time
from sklearn.metrics import roc_auc_score

save_dir = "/content/drive/MyDrive/thesis_experiments"
train = pd.read_parquet(f"{save_dir}/split_train.parquet")
val   = pd.read_parquet(f"{save_dir}/split_val.parquet")
test  = pd.read_parquet(f"{save_dir}/split_test.parquet")

string_cols = ["URL", "Domain", "TLD", "Title"]
FEATURES_C = [c for c in train.columns
              if c not in string_cols + ["label", "URLSimilarityIndex"]]   # change 1
print("Feature count (should be 49):", len(FEATURES_C))

Xtr = train[FEATURES_C].astype(np.float32).values
Xva = val[FEATURES_C].astype(np.float32).values
Xte = test[FEATURES_C].astype(np.float32).values
mn, mx = Xtr.min(axis=0), Xtr.max(axis=0)
rng = np.where((mx - mn) == 0, 1.0, mx - mn)
Xtr = (Xtr - mn) / rng
Xva = np.clip((Xva - mn) / rng, 0, 1)
Xte = np.clip((Xte - mn) / rng, 0, 1)
ytr = train["label"].values.astype(np.float32)
yva = val["label"].values.astype(np.float32)
yte = test["label"].values.astype(np.float32)
np.save(f"{save_dir}/scaler_min_C.npy", mn); np.save(f"{save_dir}/scaler_rng_C.npy", rng)  # change 2

class ADUIN(nn.Module):
    def __init__(self, n_features, d_model=128, n_heads=8, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout),
        )
        self.out = nn.Linear(32, 1)
    def forward(self, x):
        h = self.input_proj(x).unsqueeze(1)
        a, _ = self.attn(h, h, h)
        h = self.norm(h + a).squeeze(1)
        return self.out(self.mlp(h)).squeeze(-1)

device = "cuda"
frac_phish = (ytr == 0).mean()
sample_w_tr = np.where(ytr == 0, 1.0 - frac_phish, frac_phish).astype(np.float32)

def run_seed(seed, epochs=50, batch=128, patience=5):
    torch.manual_seed(seed); np.random.seed(seed)
    model = ADUIN(len(FEATURES_C)).to(device)                               # change 3
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    Xt = torch.tensor(Xtr); yt = torch.tensor(ytr); wt = torch.tensor(sample_w_tr)
    Xv = torch.tensor(Xva).to(device); yv = torch.tensor(yva).to(device)
    n = len(Xt); best_val = float("inf"); best_ep = 0; wait = 0
    ckpt = f"{save_dir}/baselineC_seed{seed}.pt"                            # change 2
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, batch):
            idx = perm[i:i+batch]
            xb, yb, wb = Xt[idx].to(device), yt[idx].to(device), wt[idx].to(device)
            opt.zero_grad()
            loss = nn.functional.binary_cross_entropy_with_logits(model(xb), yb, weight=wb)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = nn.functional.binary_cross_entropy_with_logits(model(Xv), yv).item()
        if vl < best_val - 1e-5:
            best_val, best_ep, wait = vl, ep, 0
            torch.save(model.state_dict(), ckpt)
        else:
            wait += 1
            if wait >= patience:
                break
    print(f"  seed {seed}: best epoch {best_ep}, best val loss {best_val:.5f}")

for seed in [42, 43, 44]:
    print(f"Training Baseline C seed {seed}...")
    run_seed(seed)

# Evaluation (change 4: merged into this cell)
Xt = torch.tensor(Xte).to(device)
def evaluate(path):
    model = ADUIN(len(FEATURES_C)).to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    with torch.no_grad():
        p_phish = 1 - torch.sigmoid(model(Xt)).cpu().numpy()
    pred = (p_phish >= 0.5); actual = (yte == 0)
    TP = np.sum(pred & actual); TN = np.sum(~pred & ~actual)
    FP = np.sum(pred & ~actual); FN = np.sum(~pred & actual)
    return dict(
        accuracy=(TP+TN)/len(yte),
        precision=TP/(TP+FP) if TP+FP else 0.0,
        recall_zeroday=TP/(TP+FN) if TP+FN else 0.0,
        f1=2*TP/(2*TP+FP+FN) if TP else 0.0,
        fpr=FP/(FP+TN) if FP+TN else 0.0,
        auc=roc_auc_score(actual.astype(int), p_phish),
    )

rows = []
for seed in [42, 43, 44]:
    m = evaluate(f"{save_dir}/baselineC_seed{seed}.pt"); m["seed"] = seed; rows.append(m)
res = pd.DataFrame(rows).set_index("seed")
print()
print("BASELINE C PER-SEED RESULTS (percent)")
print((res * 100).round(2).to_string())
print()
print("BASELINE C SUMMARY (percent)")
print((res.agg(["mean", "std"]) * 100).round(2).to_string())
print()
print("Reminder, Baseline F was: accuracy 99.98, precision 100.0, zero-day 99.96, FPR 0.00")
res.to_csv(f"{save_dir}/baselineC_test_results.csv")
print("Saved to baselineC_test_results.csv")

Feature count (should be 49): 49
Training Baseline C seed 42...
  seed 42: best epoch 5, best val loss 0.00327
Training Baseline C seed 43...


In [ ]:
# Step 7: Baseline U (URL-only features): train + evaluate, 3 seeds
import pandas as pd, numpy as np, torch, torch.nn as nn
from sklearn.metrics import roc_auc_score

save_dir = "/content/drive/MyDrive/thesis_experiments"
train = pd.read_parquet(f"{save_dir}/split_train.parquet")
val   = pd.read_parquet(f"{save_dir}/split_val.parquet")
test  = pd.read_parquet(f"{save_dir}/split_test.parquet")

# URL-only: computable from the URL string (plus static lookup tables),
# no page fetch required. Bank/Pay/Crypto are URL keyword flags in PhiUSIIL.
# URLSimilarityIndex excluded: leaky (Finding E) and needs a reference DB.
FEATURES_U = [
    "URLLength", "DomainLength", "IsDomainIP", "CharContinuationRate",
    "TLDLegitimateProb", "URLCharProb", "TLDLength", "NoOfSubDomain",
    "HasObfuscation", "NoOfObfuscatedChar", "ObfuscationRatio",
    "NoOfLettersInURL", "LetterRatioInURL", "NoOfDegitsInURL", "DegitRatioInURL",
    "NoOfEqualsInURL", "NoOfQMarkInURL", "NoOfAmpersandInURL",
    "NoOfOtherSpecialCharsInURL", "SpacialCharRatioInURL", "IsHTTPS",
    "Bank", "Pay", "Crypto", "HasObfuscation" if False else "URLTitleMatchScore",
]
# Correction: URLTitleMatchScore needs the page title, so it is NOT URL-only.
FEATURES_U = [f for f in FEATURES_U if f != "URLTitleMatchScore"]
print("URL-only feature count:", len(FEATURES_U))
for f in FEATURES_U: print("  ", f)

Xtr = train[FEATURES_U].astype(np.float32).values
Xva = val[FEATURES_U].astype(np.float32).values
Xte = test[FEATURES_U].astype(np.float32).values
mn, mx = Xtr.min(axis=0), Xtr.max(axis=0)
rng = np.where((mx - mn) == 0, 1.0, mx - mn)
Xtr = (Xtr - mn) / rng
Xva = np.clip((Xva - mn) / rng, 0, 1)
Xte = np.clip((Xte - mn) / rng, 0, 1)
ytr = train["label"].values.astype(np.float32)
yva = val["label"].values.astype(np.float32)
yte = test["label"].values.astype(np.float32)
np.save(f"{save_dir}/scaler_min_U.npy", mn); np.save(f"{save_dir}/scaler_rng_U.npy", rng)
with open(f"{save_dir}/features_U.txt", "w") as fh: fh.write("\n".join(FEATURES_U))

class ADUIN(nn.Module):
    def __init__(self, n_features, d_model=128, n_heads=8, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout),
        )
        self.out = nn.Linear(32, 1)
    def forward(self, x):
        h = self.input_proj(x).unsqueeze(1)
        a, _ = self.attn(h, h, h)
        h = self.norm(h + a).squeeze(1)
        return self.out(self.mlp(h)).squeeze(-1)

device = "cuda"
probe = ADUIN(len(FEATURES_U))
n_params = sum(p.numel() for p in probe.parameters())
print(f"\nBaseline U parameter count: {n_params:,} "
      f"({sum(p.numel()*p.element_size() for p in probe.parameters())/1e6:.2f} MB FP32)")

frac_phish = (ytr == 0).mean()
sample_w_tr = np.where(ytr == 0, 1.0 - frac_phish, frac_phish).astype(np.float32)

def run_seed(seed, epochs=50, batch=128, patience=5):
    torch.manual_seed(seed); np.random.seed(seed)
    model = ADUIN(len(FEATURES_U)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    Xt = torch.tensor(Xtr); yt = torch.tensor(ytr); wt = torch.tensor(sample_w_tr)
    Xv = torch.tensor(Xva).to(device); yv = torch.tensor(yva).to(device)
    n = len(Xt); best_val = float("inf"); best_ep = 0; wait = 0
    ckpt = f"{save_dir}/baselineU_seed{seed}.pt"
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, batch):
            idx = perm[i:i+batch]
            xb, yb, wb = Xt[idx].to(device), yt[idx].to(device), wt[idx].to(device)
            opt.zero_grad()
            loss = nn.functional.binary_cross_entropy_with_logits(model(xb), yb, weight=wb)
            loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            vl = nn.functional.binary_cross_entropy_with_logits(model(Xv), yv).item()
        if vl < best_val - 1e-5:
            best_val, best_ep, wait = vl, ep, 0
            torch.save(model.state_dict(), ckpt)
        else:
            wait += 1
            if wait >= patience: break
    print(f"  seed {seed}: best epoch {best_ep}, best val loss {best_val:.4f}")

for seed in [42, 43, 44]:
    print(f"Training Baseline U seed {seed}...")
    run_seed(seed)

Xt = torch.tensor(Xte).to(device)
def evaluate(path):
    model = ADUIN(len(FEATURES_U)).to(device)
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    with torch.no_grad():
        p_phish = 1 - torch.sigmoid(model(Xt)).cpu().numpy()
    pred = (p_phish >= 0.5); actual = (yte == 0)
    TP = np.sum(pred & actual); TN = np.sum(~pred & ~actual)
    FP = np.sum(pred & ~actual); FN = np.sum(~pred & actual)
    return dict(
        accuracy=(TP+TN)/len(yte),
        precision=TP/(TP+FP) if TP+FP else 0.0,
        recall_zeroday=TP/(TP+FN) if TP+FN else 0.0,
        f1=2*TP/(2*TP+FP+FN) if TP else 0.0,
        fpr=FP/(FP+TN) if FP+TN else 0.0,
        auc=roc_auc_score(actual.astype(int), p_phish),
    )

rows = []
for seed in [42, 43, 44]:
    m = evaluate(f"{save_dir}/baselineU_seed{seed}.pt"); m["seed"] = seed; rows.append(m)
res = pd.DataFrame(rows).set_index("seed")
print()
print("BASELINE U PER-SEED RESULTS (percent)")
print((res * 100).round(2).to_string())
print()
print("BASELINE U SUMMARY (percent)")
print((res.agg(["mean", "std"]) * 100).round(2).to_string())
print()
print("Comparison: F = 99.98 acc / 99.96 zd / 0.00 fpr;  C = 99.93 / 99.87 / 0.03")
res.to_csv(f"{save_dir}/baselineU_test_results.csv")
print("Saved to baselineU_test_results.csv")

In [ ]:
print(f"\nDone in {(time.time()-t0)/60:.1f} min. Saved pruning_grid_results.csv")

In [ ]:
# Step 8 (fixed): Magnitude pruning grid on Baseline U, manual masks
import pandas as pd, numpy as np, torch, torch.nn as nn, copy, time
from sklearn.metrics import roc_auc_score

save_dir = "/content/drive/MyDrive/thesis_experiments"
train = pd.read_parquet(f"{save_dir}/split_train.parquet")
val   = pd.read_parquet(f"{save_dir}/split_val.parquet")
test  = pd.read_parquet(f"{save_dir}/split_test.parquet")

FEATURES_U = open(f"{save_dir}/features_U.txt").read().split("\n")
mn = np.load(f"{save_dir}/scaler_min_U.npy"); rng = np.load(f"{save_dir}/scaler_rng_U.npy")

def prep(dfp):
    X = np.clip((dfp[FEATURES_U].astype(np.float32).values - mn) / rng, 0, 1).astype(np.float32)
    return X, dfp["label"].values.astype(np.float32)

Xtr, ytr = prep(train); Xva, yva = prep(val); Xte, yte = prep(test)

train_domains = set(train["Domain"].str.lower())
test_dom = test["Domain"].str.lower().values
phish_mask = (yte == 0)
in_train = pd.Series(test_dom).isin(train_domains).values
unseen_mask = phish_mask & ~in_train
seen_mask   = phish_mask &  in_train
print(f"Phishing test rows: {phish_mask.sum()} (unseen {unseen_mask.sum()}, seen {seen_mask.sum()})")

class ADUIN(nn.Module):
    def __init__(self, n_features, d_model=128, n_heads=8, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout),
        )
        self.out = nn.Linear(32, 1)
    def forward(self, x):
        h = self.input_proj(x).unsqueeze(1)
        a, _ = self.attn(h, h, h)
        h = self.norm(h + a).squeeze(1)
        return self.out(self.mlp(h)).squeeze(-1)

device = "cuda"
frac_phish = (ytr == 0).mean()
sample_w = np.where(ytr == 0, 1.0 - frac_phish, frac_phish).astype(np.float32)

def prunable_tensors(model):
    ts = [model.input_proj.weight, model.out.weight,
          model.attn.out_proj.weight, model.attn.in_proj_weight]
    ts += [m.weight for m in model.mlp if isinstance(m, nn.Linear)]
    return ts

def apply_global_magnitude_prune(model, amount):
    """Zero the smallest-|w| fraction 'amount' globally; return masks."""
    ts = prunable_tensors(model)
    allw = torch.cat([t.detach().abs().flatten() for t in ts])
    k = max(1, int(amount * allw.numel()))
    thr = torch.kthvalue(allw, k).values
    masks = []
    with torch.no_grad():
        for t in ts:
            m = (t.abs() > thr).float()
            t.mul_(m)
            masks.append(m)
    return masks

def reapply(model, masks):
    with torch.no_grad():
        for t, m in zip(prunable_tensors(model), masks):
            t.mul_(m)

def sparsity_of(model):
    ts = prunable_tensors(model)
    zeros = sum((t == 0).sum().item() for t in ts)
    total = sum(t.numel() for t in ts)
    return zeros / total

def finetune(model, masks, epochs=10, batch=128, patience=3):
    opt = torch.optim.Adam(model.parameters(), lr=0.0005)
    Xt = torch.tensor(Xtr); yt = torch.tensor(ytr); wt = torch.tensor(sample_w)
    Xv = torch.tensor(Xva).to(device); yv = torch.tensor(yva).to(device)
    n = len(Xt); best_val = float("inf"); best_state = None; wait = 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, batch):
            idx = perm[i:i+batch]
            xb, yb, wb = Xt[idx].to(device), yt[idx].to(device), wt[idx].to(device)
            opt.zero_grad()
            nn.functional.binary_cross_entropy_with_logits(model(xb), yb, weight=wb).backward()
            opt.step()
            reapply(model, masks)     # keep pruned weights at zero
        model.eval()
        with torch.no_grad():
            vl = nn.functional.binary_cross_entropy_with_logits(model(Xv), yv).item()
        if vl < best_val - 1e-5:
            best_val = vl; best_state = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
            if wait >= patience: break
    if best_state: model.load_state_dict(best_state)
    reapply(model, masks)
    return model

def full_eval(model):
    model.eval()
    with torch.no_grad():
        p_phish = 1 - torch.sigmoid(model(torch.tensor(Xte).to(device))).cpu().numpy()
    pred = (p_phish >= 0.5); actual = (yte == 0)
    TP = np.sum(pred & actual); TN = np.sum(~pred & ~actual)
    FP = np.sum(pred & ~actual); FN = np.sum(~pred & actual)
    return dict(
        accuracy=(TP+TN)/len(yte),
        precision=TP/(TP+FP) if TP+FP else 0.0,
        zeroday=TP/(TP+FN) if TP+FN else 0.0,
        fpr=FP/(FP+TN) if FP+TN else 0.0,
        auc=roc_auc_score(actual.astype(int), p_phish),
        zd_unseen=pred[unseen_mask].mean(),
        zd_seen=pred[seen_mask].mean(),
    )

rows = []
t0 = time.time()
for seed in [42, 43, 44]:
    base = ADUIN(len(FEATURES_U)).to(device)
    base.load_state_dict(torch.load(f"{save_dir}/baselineU_seed{seed}.pt", map_location=device))
    m = full_eval(base); m.update(seed=seed, target_sparsity=0.0, actual_sparsity=0.0); rows.append(m)
    for target_s in [0.5, 0.7, 0.9, 0.95]:
        model = copy.deepcopy(base)
        masks = apply_global_magnitude_prune(model, target_s)
        model = finetune(model, masks)
        s = sparsity_of(model)
        m = full_eval(model); m.update(seed=seed, target_sparsity=target_s, actual_sparsity=round(s, 3))
        rows.append(m)
        torch.save(model.state_dict(), f"{save_dir}/pruneU_s{int(target_s*100)}_seed{seed}.pt")
        print(f"seed {seed} target {target_s:.0%}: actual sparsity {s:.3f}, "
              f"acc {m['accuracy']*100:.2f}, zd_unseen {m['zd_unseen']*100:.2f}")

res = pd.DataFrame(rows)
print()
print("PRUNING GRID SUMMARY (percent, mean ± std over 3 seeds)")
agg = res.groupby("target_sparsity").agg(["mean", "std"])
header = "metric       " + "  ".join(f"{s:>12.0%}" for s in sorted(res["target_sparsity"].unique()))
print(header)
for col in ["accuracy", "precision", "zeroday", "fpr", "auc", "zd_unseen", "zd_seen"]:
    line = "  ".join(f"{agg.loc[s,(col,'mean')]*100:6.2f}±{agg.loc[s,(col,'std')]*100:4.2f}"
                     for s in sorted(res["target_sparsity"].unique()))
    print(f"{col:12s} {line}")
print()
n_total = sum(p.numel() for p in ADUIN(len(FEATURES_U)).parameters())
for s in [0.0, 0.5, 0.7, 0.9, 0.95]:
    nz = int(n_total * (1 - s))
    print(f"sparsity {s:.0%}: ~{nz:,} non-zero params, theoretical sparse size {nz*4/1e6:.3f} MB")
print("(dense FP32 file stays ~0.39 MB at all levels; noted for thesis)")
res.to_csv(f"{save_dir}/pruning_grid_results.csv", index=False)
print(f"\nDone in {(time.time()-t0)/60:.1f} min. Saved pruning_grid_results.csv")

In [ ]:
# Step 8b: Reprint pruning grid summary from the saved results
import pandas as pd

save_dir = "/content/drive/MyDrive/thesis_experiments"
res = pd.read_csv(f"{save_dir}/pruning_grid_results.csv")

agg = res.groupby("target_sparsity").agg(["mean", "std"])
levels = sorted(res["target_sparsity"].unique())
print("PRUNING GRID SUMMARY (percent, mean ± std over 3 seeds)")
print("metric       " + "  ".join(f"{s:>12.0%}" for s in levels))
for col in ["accuracy", "precision", "zeroday", "fpr", "auc", "zd_unseen", "zd_seen"]:
    line = "  ".join(f"{agg.loc[s,(col,'mean')]*100:6.2f}±{agg.loc[s,(col,'std')]*100:4.2f}"
                     for s in levels)
    print(f"{col:12s} {line}")
print()
print("Actual sparsity achieved per target:")
print(res.groupby("target_sparsity")["actual_sparsity"].mean().round(3).to_string())

In [ ]:
# Step 9: Iterative (gradual) magnitude pruning on Baseline U
import pandas as pd, numpy as np, torch, torch.nn as nn, copy, time
from sklearn.metrics import roc_auc_score

save_dir = "/content/drive/MyDrive/thesis_experiments"
train = pd.read_parquet(f"{save_dir}/split_train.parquet")
val   = pd.read_parquet(f"{save_dir}/split_val.parquet")
test  = pd.read_parquet(f"{save_dir}/split_test.parquet")
FEATURES_U = open(f"{save_dir}/features_U.txt").read().split("\n")
mn = np.load(f"{save_dir}/scaler_min_U.npy"); rng = np.load(f"{save_dir}/scaler_rng_U.npy")

def prep(dfp):
    X = np.clip((dfp[FEATURES_U].astype(np.float32).values - mn) / rng, 0, 1).astype(np.float32)
    return X, dfp["label"].values.astype(np.float32)
Xtr, ytr = prep(train); Xva, yva = prep(val); Xte, yte = prep(test)

train_domains = set(train["Domain"].str.lower())
in_train = pd.Series(test["Domain"].str.lower().values).isin(train_domains).values
phish_mask = (yte == 0)
unseen_mask = phish_mask & ~in_train
seen_mask = phish_mask & in_train

class ADUIN(nn.Module):
    def __init__(self, n_features, d_model=128, n_heads=8, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout),
        )
        self.out = nn.Linear(32, 1)
    def forward(self, x):
        h = self.input_proj(x).unsqueeze(1)
        a, _ = self.attn(h, h, h)
        h = self.norm(h + a).squeeze(1)
        return self.out(self.mlp(h)).squeeze(-1)

device = "cuda"
frac_phish = (ytr == 0).mean()
sample_w = np.where(ytr == 0, 1.0 - frac_phish, frac_phish).astype(np.float32)

def prunable_tensors(model):
    ts = [model.input_proj.weight, model.out.weight,
          model.attn.out_proj.weight, model.attn.in_proj_weight]
    ts += [m.weight for m in model.mlp if isinstance(m, nn.Linear)]
    return ts

def prune_to(model, amount):
    ts = prunable_tensors(model)
    allw = torch.cat([t.detach().abs().flatten() for t in ts])
    k = max(1, int(amount * allw.numel()))
    thr = torch.kthvalue(allw, k).values
    masks = []
    with torch.no_grad():
        for t in ts:
            m = (t.abs() > thr).float()
            t.mul_(m); masks.append(m)
    return masks

def reapply(model, masks):
    with torch.no_grad():
        for t, m in zip(prunable_tensors(model), masks):
            t.mul_(m)

def finetune(model, masks, epochs, lr=0.0005, batch=128, patience=3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xt = torch.tensor(Xtr); yt = torch.tensor(ytr); wt = torch.tensor(sample_w)
    Xv = torch.tensor(Xva).to(device); yv = torch.tensor(yva).to(device)
    n = len(Xt); best_val = float("inf"); best_state = None; wait = 0
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, batch):
            idx = perm[i:i+batch]
            xb, yb, wb = Xt[idx].to(device), yt[idx].to(device), wt[idx].to(device)
            opt.zero_grad()
            nn.functional.binary_cross_entropy_with_logits(model(xb), yb, weight=wb).backward()
            opt.step()
            reapply(model, masks)
        model.eval()
        with torch.no_grad():
            vl = nn.functional.binary_cross_entropy_with_logits(model(Xv), yv).item()
        if vl < best_val - 1e-5:
            best_val = vl; best_state = copy.deepcopy(model.state_dict()); wait = 0
        else:
            wait += 1
            if wait >= patience: break
    if best_state: model.load_state_dict(best_state)
    reapply(model, masks)
    return model

def full_eval(model):
    model.eval()
    with torch.no_grad():
        p_phish = 1 - torch.sigmoid(model(torch.tensor(Xte).to(device))).cpu().numpy()
    pred = (p_phish >= 0.5); actual = (yte == 0)
    TP = np.sum(pred & actual); TN = np.sum(~pred & ~actual)
    FP = np.sum(pred & ~actual); FN = np.sum(~pred & actual)
    return dict(
        accuracy=(TP+TN)/len(yte),
        precision=TP/(TP+FP) if TP+FP else 0.0,
        zeroday=TP/(TP+FN) if TP+FN else 0.0,
        fpr=FP/(FP+TN) if FP+TN else 0.0,
        auc=roc_auc_score(actual.astype(int), p_phish),
        zd_unseen=pred[unseen_mask].mean(),
        zd_seen=pred[seen_mask].mean(),
        collapsed=int(pred.sum() == 0 or pred.mean() > 0.999),
    )

# Gradual schedule: intermediate stops with fine-tuning at each stop
SCHEDULES = {0.70: [0.5, 0.7], 0.90: [0.5, 0.7, 0.85, 0.90], 0.95: [0.5, 0.7, 0.85, 0.90, 0.95]}

rows = []
t0 = time.time()
for seed in [42, 43, 44]:
    for target, steps in SCHEDULES.items():
        model = ADUIN(len(FEATURES_U)).to(device)
        model.load_state_dict(torch.load(f"{save_dir}/baselineU_seed{seed}.pt", map_location=device))
        for s in steps:
            masks = prune_to(model, s)
            model = finetune(model, masks, epochs=5)
        model = finetune(model, masks, epochs=10)   # final consolidation
        m = full_eval(model); m.update(seed=seed, target_sparsity=target)
        rows.append(m)
        torch.save(model.state_dict(), f"{save_dir}/pruneU_iter_s{int(target*100)}_seed{seed}.pt")
        print(f"seed {seed} target {target:.0%}: acc {m['accuracy']*100:.2f}, "
              f"zd_unseen {m['zd_unseen']*100:.2f}, collapsed={m['collapsed']}")

res = pd.DataFrame(rows)
print()
print("ITERATIVE PRUNING, PER-SEED (percent)")
show = res.copy()
for c in ["accuracy","precision","zeroday","fpr","auc","zd_unseen","zd_seen"]:
    show[c] = (show[c]*100).round(2)
print(show[["seed","target_sparsity","accuracy","precision","zeroday","fpr","auc","zd_unseen","zd_seen","collapsed"]].to_string(index=False))
print()
print("Collapse count by sparsity:", res.groupby("target_sparsity")["collapsed"].sum().to_dict())
ok = res[res["collapsed"] == 0]
if len(ok):
    print()
    print("SUMMARY OVER NON-COLLAPSED RUNS (percent, mean ± std)")
    agg = ok.groupby("target_sparsity").agg(["mean","std"])
    for col in ["accuracy","zeroday","fpr","auc","zd_unseen","zd_seen"]:
        line = "  ".join(f"{agg.loc[s,(col,'mean')]*100:6.2f}±{(agg.loc[s,(col,'std')]*100 if not np.isnan(agg.loc[s,(col,'std')]) else 0):4.2f}"
                         for s in sorted(ok["target_sparsity"].unique()))
        print(f"{col:10s} {line}")
res.to_csv(f"{save_dir}/pruning_iterative_results.csv", index=False)
print(f"\nDone in {(time.time()-t0)/60:.1f} min. Saved pruning_iterative_results.csv")

In [ ]:
# Step 10: Post-training INT8 dynamic quantization + CPU latency measurement
import pandas as pd, numpy as np, torch, torch.nn as nn, time, os
from sklearn.metrics import roc_auc_score

save_dir = "/content/drive/MyDrive/thesis_experiments"
test = pd.read_parquet(f"{save_dir}/split_test.parquet")
train = pd.read_parquet(f"{save_dir}/split_train.parquet")
FEATURES_U = open(f"{save_dir}/features_U.txt").read().split("\n")
mn = np.load(f"{save_dir}/scaler_min_U.npy"); rng = np.load(f"{save_dir}/scaler_rng_U.npy")
Xte = np.clip((test[FEATURES_U].astype(np.float32).values - mn) / rng, 0, 1).astype(np.float32)
yte = test["label"].values.astype(np.float32)

train_domains = set(train["Domain"].str.lower())
in_train = pd.Series(test["Domain"].str.lower().values).isin(train_domains).values
phish_mask = (yte == 0)
unseen_mask = phish_mask & ~in_train

class ADUIN(nn.Module):
    def __init__(self, n_features, d_model=128, n_heads=8, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout),
        )
        self.out = nn.Linear(32, 1)
    def forward(self, x):
        h = self.input_proj(x).unsqueeze(1)
        a, _ = self.attn(h, h, h)
        h = self.norm(h + a).squeeze(1)
        return self.out(self.mlp(h)).squeeze(-1)

def eval_cpu(model):
    model.eval()
    with torch.no_grad():
        p_phish = 1 - torch.sigmoid(model(torch.tensor(Xte))).numpy()
    pred = (p_phish >= 0.5); actual = (yte == 0)
    TP = np.sum(pred & actual); TN = np.sum(~pred & ~actual)
    FP = np.sum(pred & ~actual); FN = np.sum(~pred & actual)
    return dict(
        accuracy=(TP+TN)/len(yte),
        precision=TP/(TP+FP) if TP+FP else 0.0,
        zeroday=TP/(TP+FN) if TP+FN else 0.0,
        fpr=FP/(FP+TN) if FP+TN else 0.0,
        auc=roc_auc_score(actual.astype(int), p_phish),
        zd_unseen=pred[unseen_mask].mean(),
    )

def file_size_mb(model, path):
    torch.save(model.state_dict(), path)
    return os.path.getsize(path) / 1e6

def latency(model, n_single=300, batch_size=512, n_batch=30):
    model.eval()
    x1 = torch.tensor(Xte[:1]); xb = torch.tensor(Xte[:batch_size])
    with torch.no_grad():
        for _ in range(20): model(x1)          # warmup
        t0 = time.perf_counter()
        for _ in range(n_single): model(x1)
        per_url_ms = (time.perf_counter() - t0) / n_single * 1000
        for _ in range(5): model(xb)           # warmup
        t0 = time.perf_counter()
        for _ in range(n_batch): model(xb)
        batch_ms = (time.perf_counter() - t0) / n_batch * 1000
    return per_url_ms, batch_ms, batch_ms / batch_size

rows = []
for seed in [42, 43, 44]:
    fp32 = ADUIN(len(FEATURES_U))
    fp32.load_state_dict(torch.load(f"{save_dir}/baselineU_seed{seed}.pt", map_location="cpu"))

    int8 = torch.ao.quantization.quantize_dynamic(fp32, {nn.Linear}, dtype=torch.qint8)

    if seed == 42:
        print("Quantized module types (seed 42):")
        for name, mod in int8.named_modules():
            if "Linear" in type(mod).__name__:
                print(f"  {name}: {type(mod).__name__}")
        print()

    for tag, model in [("FP32", fp32), ("INT8-PTQ", int8)]:
        m = eval_cpu(model)
        size = file_size_mb(model, f"{save_dir}/tmp_{tag}_{seed}.pt")
        pu, bm, pb = latency(model)
        m.update(seed=seed, model=tag, size_mb=round(size, 3),
                 ms_per_url=round(pu, 3), ms_per_batch512=round(bm, 1),
                 ms_per_url_batched=round(pb, 4))
        rows.append(m)
        print(f"seed {seed} {tag}: acc {m['accuracy']*100:.2f}, zd_unseen {m['zd_unseen']*100:.2f}, "
              f"size {size:.3f} MB, {pu:.2f} ms/URL single, {pb:.4f} ms/URL batched")

res = pd.DataFrame(rows)
print()
print("QUANTIZATION SUMMARY (mean over 3 seeds)")
agg = res.groupby("model").mean(numeric_only=True)
cols = ["accuracy", "precision", "zeroday", "fpr", "auc", "zd_unseen"]
out = (agg[cols] * 100).round(2)
out[["size_mb", "ms_per_url", "ms_per_batch512", "ms_per_url_batched"]] = \
    agg[["size_mb", "ms_per_url", "ms_per_batch512", "ms_per_url_batched"]].round(3)
print(out.to_string())
print()
print("Reference: paper claims '210 ms under high load' (per-URL vs per-batch ambiguous).")
res.to_csv(f"{save_dir}/quantization_ptq_results.csv", index=False)
print("Saved quantization_ptq_results.csv")

In [ ]:
# Recovery: remount Drive and restore splits
from google.colab import drive
drive.mount("/content/drive")

import os
import pandas as pd

save_dir = "/content/drive/MyDrive/thesis_experiments"
print("Contents of thesis_experiments:", os.listdir(save_dir))

needed = ["split_train.parquet", "split_val.parquet", "split_test.parquet"]
missing = [f for f in needed if not os.path.exists(f"{save_dir}/{f}")]

if not missing:
    print("All split files present. Nothing to rebuild.")
else:
    print("Missing:", missing, "- rebuilding from full dataset (deterministic, same result).")
    df = pd.read_parquet(f"{save_dir}/phiusiil_full.parquet")
    n = len(df)
    i_train, i_val = int(n * 0.70), int(n * 0.85)
    df.iloc[:i_train].to_parquet(f"{save_dir}/split_train.parquet")
    df.iloc[i_train:i_val].to_parquet(f"{save_dir}/split_val.parquet")
    df.iloc[i_val:].to_parquet(f"{save_dir}/split_test.parquet")
    print("Splits rebuilt and saved.")

# Quick verification
tr = pd.read_parquet(f"{save_dir}/split_train.parquet")
print("Train rows:", len(tr), "(expect 165056)")

In [ ]:
# Step 11A: distillation setup
import pandas as pd, numpy as np, torch, torch.nn as nn, copy, time, os
from sklearn.metrics import roc_auc_score

save_dir = "/content/drive/MyDrive/thesis_experiments"
train = pd.read_parquet(f"{save_dir}/split_train.parquet")
val   = pd.read_parquet(f"{save_dir}/split_val.parquet")
test  = pd.read_parquet(f"{save_dir}/split_test.parquet")
FEATURES_U = open(f"{save_dir}/features_U.txt").read().split("\n")
mn = np.load(f"{save_dir}/scaler_min_U.npy")
rng = np.load(f"{save_dir}/scaler_rng_U.npy")

def prep(d):
    X = np.clip((d[FEATURES_U].astype(np.float32).values - mn) / rng, 0, 1)
    return X.astype(np.float32), d["label"].values.astype(np.float32)

Xtr, ytr = prep(train)
Xva, yva = prep(val)
Xte, yte = prep(test)

train_domains = set(train["Domain"].str.lower())
in_train = pd.Series(test["Domain"].str.lower().values).isin(train_domains).values
phish_mask = (yte == 0)
unseen_mask = phish_mask & ~in_train
print("Setup A done. Unseen-domain phishing rows:", unseen_mask.sum())

In [ ]:

# Step 11B: model classes
class ADUIN(nn.Module):
    def __init__(self, n_features, d_model=128, n_heads=8, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(
            nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(dropout),
        )
        self.out = nn.Linear(32, 1)

    def forward(self, x):
        h = self.input_proj(x).unsqueeze(1)
        a, _ = self.attn(h, h, h)
        h = self.norm(h + a).squeeze(1)
        return self.out(self.mlp(h)).squeeze(-1)

STUDENT_DIMS = {"S-192": [128, 64, 32], "S-64": [64, 32], "S-16": [16, 8], "S-LIN": []}

class Student(nn.Module):
    def __init__(self, name, n_in):
        super().__init__()
        layers = []
        prev = n_in
        for d in STUDENT_DIMS[name]:
            layers.append(nn.Linear(prev, d))
            layers.append(nn.ReLU())
            prev = d
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)

print("Setup B done. Students:", list(STUDENT_DIMS))

In [ ]:
# Step 11C: distillation loop
device = "cuda"
frac_phish = (ytr == 0).mean()
sample_w = np.where(ytr == 0, 1.0 - frac_phish, frac_phish).astype(np.float32)
T, ALPHA = 2.0, 0.5

def distill(student, teacher, epochs=30, batch=128, patience=4, lr=0.001):
    teacher.eval()
    with torch.no_grad():
        soft = torch.sigmoid(teacher(torch.tensor(Xtr).to(device)) / T).cpu()
    opt = torch.optim.Adam(student.parameters(), lr=lr)
    Xt = torch.tensor(Xtr)
    yt = torch.tensor(ytr)
    wt = torch.tensor(sample_w)
    Xv = torch.tensor(Xva).to(device)
    yv = torch.tensor(yva).to(device)
    n = len(Xt)
    best = float("inf")
    best_state = None
    wait = 0
    for ep in range(epochs):
        student.train()
        perm = torch.randperm(n)
        for i in range(0, n, batch):
            idx = perm[i:i+batch]
            xb = Xt[idx].to(device)
            yb = yt[idx].to(device)
            wb = wt[idx].to(device)
            sb = soft[idx].to(device)
            opt.zero_grad()
            logit = student(xb)
            hard = nn.functional.binary_cross_entropy_with_logits(logit, yb, weight=wb)
            sl = nn.functional.binary_cross_entropy_with_logits(logit / T, sb)
            ((1 - ALPHA) * hard + ALPHA * (T * T) * sl).backward()
            opt.step()
        student.eval()
        with torch.no_grad():
            vl = nn.functional.binary_cross_entropy_with_logits(student(Xv), yv).item()
        if vl < best - 1e-5:
            best = vl
            best_state = copy.deepcopy(student.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    if best_state:
        student.load_state_dict(best_state)
    return student

def full_eval(model):
    model.eval()
    with torch.no_grad():
        p = 1 - torch.sigmoid(model(torch.tensor(Xte).to(device))).cpu().numpy()
    pred = (p >= 0.5)
    act = (yte == 0)
    TP = np.sum(pred & act); TN = np.sum(~pred & ~act)
    FP = np.sum(pred & ~act); FN = np.sum(~pred & act)
    return dict(
        accuracy=(TP+TN)/len(yte),
        precision=TP/(TP+FP) if TP+FP else 0.0,
        zeroday=TP/(TP+FN) if TP+FN else 0.0,
        fpr=FP/(FP+TN) if FP+TN else 0.0,
        auc=roc_auc_score(act.astype(int), p),
        zd_unseen=pred[unseen_mask].mean(),
        collapsed=int(pred.sum() == 0 or pred.mean() > 0.999),
    )

def cpu_latency(model, n=300):
    mc = copy.deepcopy(model).cpu().eval()
    x1 = torch.tensor(Xte[:1])
    with torch.no_grad():
        for _ in range(20):
            mc(x1)
        t0 = time.perf_counter()
        for _ in range(n):
            mc(x1)
    return (time.perf_counter() - t0) / n * 1000

rows = []
t0 = time.time()
for seed in [42, 43, 44]:
    torch.manual_seed(seed)
    np.random.seed(seed)
    teacher = ADUIN(len(FEATURES_U)).to(device)
    teacher.load_state_dict(torch.load(f"{save_dir}/baselineU_seed{seed}.pt", map_location=device))
    for name in ["S-192", "S-64", "S-16", "S-LIN"]:
        st = Student(name, len(FEATURES_U)).to(device)
        npar = sum(p.numel() for p in st.parameters())
        st = distill(st, teacher)
        m = full_eval(st)
        path = f"{save_dir}/student_{name}_seed{seed}.pt"
        torch.save(st.state_dict(), path)
        m.update(seed=seed, student=name, params=npar,
                 size_mb=round(os.path.getsize(path)/1e6, 4),
                 ms_per_url=round(cpu_latency(st), 3))
        rows.append(m)
        print(f"seed {seed} {name}: {npar:,} params, acc {m['accuracy']*100:.2f}, "
              f"zd_unseen {m['zd_unseen']*100:.2f}, {m['ms_per_url']:.3f} ms/URL, "
              f"collapsed={m['collapsed']}")

res_dist = pd.DataFrame(rows)
res_dist.to_csv(f"{save_dir}/distillation_results.csv", index=False)
print(f"\nDone in {(time.time()-t0)/60:.1f} min. Saved distillation_results.csv")

In [ ]:
# Step 11D: distillation summary table vs teacher
res_dist = pd.read_csv(f"{save_dir}/distillation_results.csv")
agg = res_dist.groupby("student").agg(["mean", "std"])

print(f"{'model':8s} {'params':>8s} {'ratio':>7s} {'size MB':>8s} {'ms/URL':>7s} "
      f"{'accuracy':>14s} {'zeroday':>14s} {'zd_unseen':>14s} {'fpr':>11s}")
print(f"{'teacher':8s} {96385:>8,} {'1.0x':>7s} {0.391:>8.4f} {0.879:>7.3f} "
      f"{'99.66±0.02':>14s} {'99.25±0.06':>14s} {'99.53±0.04':>14s} {'0.03±0.03':>11s}")
for s in ["S-192", "S-64", "S-16", "S-LIN"]:
    a = agg.loc[s]
    p = int(a[("params", "mean")])
    print(f"{s:8s} {p:>8,} {96385/p:>6.1f}x {a[('size_mb','mean')]:>8.4f} "
          f"{a[('ms_per_url','mean')]:>7.3f} "
          f"{a[('accuracy','mean')]*100:>7.2f}±{a[('accuracy','std')]*100:<6.2f} "
          f"{a[('zeroday','mean')]*100:>7.2f}±{a[('zeroday','std')]*100:<6.2f} "
          f"{a[('zd_unseen','mean')]*100:>7.2f}±{a[('zd_unseen','std')]*100:<6.2f} "
          f"{a[('fpr','mean')]*100:>5.2f}±{a[('fpr','std')]*100:<5.2f}")

print()
print("Speedup vs teacher (CPU, single URL):")
for s in ["S-192", "S-64", "S-16", "S-LIN"]:
    print(f"  {s}: {0.879 / agg.loc[s, ('ms_per_url','mean')]:.1f}x faster")
print()
print("Published ADUIN reference: 95.0 acc, 92.0 zero-day, 3.5 FPR, 210 ms")

In [ ]:
# Step 12: Best-combination (S-16 + INT8), pruned-model latency, final Pareto table
import pandas as pd, numpy as np, torch, torch.nn as nn, copy, time, os
from sklearn.metrics import roc_auc_score

rows12 = []

def eval_cpu_model(model):
    model.eval()
    with torch.no_grad():
        p = 1 - torch.sigmoid(model(torch.tensor(Xte))).numpy()
    pred = (p >= 0.5); act = (yte == 0)
    TP = np.sum(pred & act); TN = np.sum(~pred & ~act)
    FP = np.sum(pred & ~act); FN = np.sum(~pred & act)
    return dict(accuracy=(TP+TN)/len(yte),
                precision=TP/(TP+FP) if TP+FP else 0.0,
                zeroday=TP/(TP+FN) if TP+FN else 0.0,
                fpr=FP/(FP+TN) if FP+TN else 0.0,
                auc=roc_auc_score(act.astype(int), p),
                zd_unseen=pred[unseen_mask].mean())

def lat_cpu(model, n=300):
    m = copy.deepcopy(model).cpu().eval()
    x1 = torch.tensor(Xte[:1])
    with torch.no_grad():
        for _ in range(20): m(x1)
        t0 = time.perf_counter()
        for _ in range(n): m(x1)
    return (time.perf_counter() - t0) / n * 1000

for seed in [42, 43, 44]:
    # (a) S-16 + INT8 dynamic quantization
    s16 = Student("S-16", len(FEATURES_U))
    s16.load_state_dict(torch.load(f"{save_dir}/student_S-16_seed{seed}.pt", map_location="cpu"))
    s16q = torch.ao.quantization.quantize_dynamic(s16, {nn.Linear}, dtype=torch.qint8)
    m = eval_cpu_model(s16q)
    path = f"{save_dir}/student_S16_int8_seed{seed}.pt"
    torch.save(s16q.state_dict(), path)
    m.update(seed=seed, config="S-16 + INT8",
             size_mb=round(os.path.getsize(path)/1e6, 4), ms_per_url=round(lat_cpu(s16q), 3))
    rows12.append(m)
    print(f"seed {seed} S-16+INT8: acc {m['accuracy']*100:.2f}, zd_unseen {m['zd_unseen']*100:.2f}, "
          f"{m['size_mb']} MB, {m['ms_per_url']} ms/URL")

    # (b) Iteratively pruned 90% teacher, CPU latency + metrics
    pr = ADUIN(len(FEATURES_U))
    pr.load_state_dict(torch.load(f"{save_dir}/pruneU_iter_s90_seed{seed}.pt", map_location="cpu"))
    m = eval_cpu_model(pr)
    nz = sum(int((p != 0).sum()) for p in pr.parameters())
    m.update(seed=seed, config="Pruned 90% (dense storage)",
             size_mb=0.391, ms_per_url=round(lat_cpu(pr), 3))
    rows12.append(m)
    print(f"seed {seed} pruned90: acc {m['accuracy']*100:.2f}, {nz:,} non-zero params, "
          f"{m['ms_per_url']} ms/URL")

res12 = pd.DataFrame(rows12)
res12.to_csv(f"{save_dir}/best_combination_results.csv", index=False)

print()
print("=" * 108)
print("FINAL PARETO TABLE (mean ± std over 3 seeds)")
print("=" * 108)
print(f"{'configuration':28s} {'params':>9s} {'size MB':>8s} {'ms/URL':>7s} "
      f"{'accuracy':>13s} {'zero-day':>13s} {'zd_unseen':>13s} {'FPR':>11s}")

def line(name, params, size, ms, acc, zd, zdu, fpr):
    print(f"{name:28s} {params:>9s} {size:>8s} {ms:>7s} {acc:>13s} {zd:>13s} {zdu:>13s} {fpr:>11s}")

line("ADUIN published", "n/r", "n/r", "210*", "95.00", "92.00", "n/r", "3.50")
line("Baseline F (50 feat)", "99,713", "0.399", "n/m", "99.98±0.00", "99.96±0.00", "n/m", "0.00±0.00")
line("Baseline C (49 feat)", "99,673", "0.399", "n/m", "99.93±0.02", "99.87±0.03", "n/m", "0.03±0.02")
line("Baseline U (24 feat, teacher)", "96,385", "0.391", "0.879", "99.66±0.02", "99.25±0.06", "99.53±0.04", "0.03±0.03")
line("U + prune 90% (iterative)", "9,639 nz", "0.391", "see out", "99.72±0.01", "99.38±0.01", "99.64±0.01", "0.03±0.02")
line("U + INT8 PTQ", "96,385", "0.306", "1.440", "99.66±0.00", "99.26±0.01", "99.54±0.01", "0.03±0.00")
line("Distilled S-64", "3,713", "0.0181", "0.066", "99.68±0.02", "99.31±0.03", "99.57±0.03", "0.04±0.03")
line("Distilled S-16", "545", "0.0054", "0.049", "99.66±0.00", "99.26±0.01", "99.53±0.01", "0.04±0.01")
line("Distilled S-LIN", "25", "0.0021", "0.015", "99.00±0.02", "98.47±0.04", "98.71±0.03", "0.61±0.06")
agg12 = res12.groupby("config").agg(["mean", "std"])
a = agg12.loc["S-16 + INT8"]
line("S-16 + INT8 (best combo)", "545", f"{a[('size_mb','mean')]:.4f}",
     f"{a[('ms_per_url','mean')]:.3f}",
     f"{a[('accuracy','mean')]*100:.2f}±{a[('accuracy','std')]*100:.2f}",
     f"{a[('zeroday','mean')]*100:.2f}±{a[('zeroday','std')]*100:.2f}",
     f"{a[('zd_unseen','mean')]*100:.2f}±{a[('zd_unseen','std')]*100:.2f}",
     f"{a[('fpr','mean')]*100:.2f}±{a[('fpr','std')]*100:.2f}")
print("=" * 108)
print("* paper's 210 ms is ambiguous (per-URL vs per-batch) and incompatible with measured inference;")
print("  n/r = not reported in paper, n/m = not measured (GPU-trained baselines, CPU latency measured for U only)")
print()
p90 = agg12.loc["Pruned 90% (dense storage)"]
print(f"Pruned-90% CPU latency measured: {p90[('ms_per_url','mean')]:.3f} ms/URL "
      f"(dense storage, so no speed or size gain expected)")
print("Saved best_combination_results.csv")